In [0]:
%pip install yfinance
%pip install pandas

In [0]:
import yfinance as yf
import pandas as pd
from datetime import timedelta
from pyspark.sql.functions import max as spark_max
from datetime import timedelta

from pyspark.sql.functions import (
    col,
    current_timestamp,
    max as spark_max
)

# -----------------------------
# Get tickers from market table
# -----------------------------
tickers = (
    spark.table(
        "finance_intelligence_hub.bronze.market_activity_raw"
    )
    .select("ticker")
    .distinct()
    .toPandas()["ticker"]
    .tolist()
)

# -----------------------------
# Check if stock table exists
# -----------------------------
tables = (
    spark.sql(
        "SHOW TABLES IN finance_intelligence_hub.bronze"
    )
    .select("tableName")
    .toPandas()["tableName"]
    .tolist()
)

if "stock_prices_raw" in tables:

    last_date = (
        spark.table(
            "finance_intelligence_hub.bronze.stock_prices_raw"
        )
        .agg(
            spark_max("date").alias("max_date")
        )
        .collect()[0]["max_date"]
    )

    if last_date:
        start_date = (
            last_date +
            timedelta(days=1)
        ).strftime("%Y-%m-%d")
    else:
        start_date = "2022-01-01"

else:
    start_date = "2022-01-01"

print(f"Loading from {start_date}")

# ====================================
# Download new records only
# ====================================

all_data = []

for ticker in tickers:

    try:

        df = yf.download(
    ticker,
    start=start_date,
    auto_adjust=False
    )

        if not df.empty:

            df.reset_index(inplace=True)

            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.droplevel(1)

            df["Ticker"] = ticker

            all_data.append(df)

    except Exception as e:
        print(f"Error with {ticker}: {e}")

# ====================================
# Stop if no new data
# ====================================

if len(all_data) == 0:
    print("No new records found.")

else:

    stocks_data = pd.concat(
        all_data,
        ignore_index=True
    )

    stocks_data.columns = (
        stocks_data.columns
        .str.strip()
        .str.replace(" ", "_")
    )

    stocks_data = stocks_data.rename(columns={
        "Date": "date",
        "Adj_Close": "adj_close",
        "Close": "close",
        "High": "high",
        "Low": "low",
        "Open": "open",
        "Volume": "volume",
        "Ticker": "ticker"
    })

    spark_df = spark.createDataFrame(
        stocks_data
    )

    spark_df = (
        spark_df
        .withColumn(
            "date",
            col("date").cast("date")
        )
        .withColumn(
            "adj_close",
            col("adj_close")
            .cast("decimal(18,4)")
        )
        .withColumn(
            "close",
            col("close")
            .cast("decimal(18,4)")
        )
        .withColumn(
            "high",
            col("high")
            .cast("decimal(18,4)")
        )
        .withColumn(
            "low",
            col("low")
            .cast("decimal(18,4)")
        )
        .withColumn(
            "open",
            col("open")
            .cast("decimal(18,4)")
        )
        .withColumn(
            "volume",
            col("volume")
            .cast("bigint")
        )
        .withColumn(
            "ticker",
            col("ticker")
            .cast("string")
        )
        .withColumn(
            "dwh_loaded_at",
            current_timestamp()
        )
    )

    mode = (
        "append"
        if "stock_prices_raw" in tables
        else "overwrite"
    )

    spark_df.write \
        .format("delta") \
        .mode(mode) \
        .saveAsTable(
            "finance_intelligence_hub.bronze.stock_prices_raw"
        )
    print(
        f"Inserted {spark_df.count()} rows."
    )

In [0]:
%sql
Select count(*) from finance_intelligence_hub.bronze.stock_prices_raw